# PIQA

优点：

- 高度结构化数据，问题+回答简短，几乎不需要预处理
- 二选一问题 -> 二分类模型
- 记录充足：训练集有 16113 条记录

缺点：

- 问题简短，语料可能不足
- 选项间相似度高，有时只有一个单词的差异

In [2]:
from pathlib import Path

BASE_PATH = Path('./PIQA')

train_data_path = BASE_PATH / 'train.jsonl'
train_label_path = BASE_PATH / 'train-labels.lst'

test_data_path = BASE_PATH / 'test.jsonl'
test_label_path = BASE_PATH / 'test-labels.lst'

In [3]:
import pandas as pd

train_dataset = pd.read_json(train_data_path, lines=True, encoding="utf-8")
train_labels = pd.read_csv(train_label_path, header=None, names=["label"])

train_dataset = pd.concat([train_dataset.reset_index(drop=True), train_labels], axis=1)

test_dataset = pd.read_json(test_data_path, lines=True, encoding="utf-8")
test_labels = pd.read_csv(test_label_path, header=None, names=["label"])

test_dataset = pd.concat([test_dataset.reset_index(drop=True), test_labels], axis=1)

print(train_dataset.shape)
print(test_dataset.shape)

(16113, 4)
(1838, 4)


In [4]:
train_dataset.head(5)

,goal,sol1,sol2,label
0,"When boiling butter, when it's ready, you can",Pour it onto a plate,Pour it into a jar,1
1,"To permanently attach metal legs to a chair, y...",Weld the metal together to get it to stay firm...,Nail the metal together to get it to stay firm...,0
2,how do you indent something?,leave a space before starting the writing,press the spacebar,0
3,how do you shake something?,move it up and down and side to side quickly.,stir it very quickly.,0
4,Clean tires,"Pour water, cape off caked on dirt. Use speed...","Pour water, scrape off caked on dirt. Use a st...",1


In [5]:
train_dataset.head(10).to_csv('piqa_sample.csv')

# MultiRC

缺点：

- 多层 JSON 嵌套，解析困难
- 问题包含 html 标签，需要额外清洗步骤
- 存在一个问题对应多个正确选项的情况 -> 需要类似 SQL Query 的机制

In [ ]:
import json
from pathlib import Path

BASE_PATH = Path('./MultiRC')
multirc_train_path = BASE_PATH / 'train_456-fixedIds.json'
multirc_test_path = BASE_PATH / 'test_83-fixedIds.json'

with multirc_train_path.open(encoding="utf-8") as f:
    multirc_train = json.load(f)

with multirc_test_path.open(encoding="utf-8") as f:
    multirc_test = json.load(f)


In [46]:
from pprint import pprint

print(multirc_train.keys()) # dict_keys(['data'])

print(len(multirc_train['data']))   # 456

# pprint(multirc_train['data'][0])

pprint(multirc_train['data'][0].keys()) # dict_keys(['paragraph', 'id'])

print(multirc_train['data'][0]['paragraph'].keys()) # dict_keys(['text', 'questions'])

print(len(multirc_train['data'][0]['paragraph']['questions']))  # 12

pprint(multirc_train['data'][0]['paragraph']['questions'][0])

dict_keys(['data'])
456
dict_keys(['paragraph', 'id'])
dict_keys(['text', 'questions'])
12
{'answers': [{'isAnswer': True, 'scores': {}, 'text': 'Yes'},
             {'isAnswer': True, 'scores': {}, 'text': 'Uphold, and continue'},
             {'isAnswer': False, 'scores': {}, 'text': 'No'}],
 'idx': '0',
 'multisent': True,
 'question': 'Does the author claim the animated films message is that the NRA '
             'upholds racism?',
 'sentences_used': [1, 2, 3, 5]}


# ReClor

优点：

- 每个问题信息充足，包含背景信息 (Context 字段)
- 四选一问题 -> 四分类模型

缺点：

- 训练集只有 4638 条记录

In [29]:
import json
from pathlib import Path

BASE_PATH = Path('./ReClor')
reclor_train_path = BASE_PATH / 'train.json'
reclor_test_path = BASE_PATH / 'test.json'

with reclor_train_path.open(encoding="utf-8") as f:
    reclor_train = json.load(f)

with reclor_test_path.open(encoding="utf-8") as f:
    reclor_test = json.load(f)

In [62]:
from pprint import pprint

#pprint(reclor_train[0])
pprint(reclor_train[2])

{'answers': ['If Marge goes to the bank today, Lauren will not cash her check '
             'tomorrow. Marge will not wash her car unless it is sunny. '
             'However, it is sunny, so Marge will wash her car and go shopping '
             'with Lauren.',
             'Kevin will wash his car tomorrow only if Brittany has to go '
             'visit her grandmother. Unless Aunt Susan has to run errands, '
             'Brittany will not have to go visit her grandmother. Since Aunt '
             'Susan does not have to run errands, Kevin will not wash his car '
             'tomorrow.',
             'Renee will do her homework tonight if there is nothing good on '
             'television and if her neighbors do not have a party. Although, '
             'there is something good on television; her neighbors are also '
             'having a party. Ttherefore, Renee will attend the party.',
             'Maddie will plan a picnic only if one of her friends, Lisa or '
           

In [60]:
df_reclor_train = pd.DataFrame(reclor_train)

answers_df = pd.DataFrame(
    df_reclor_train["answers"].tolist(),
    columns=["answer_a", "answer_b", "answer_c", "answer_d"]
)

df_reclor_train = pd.concat(
    [df_reclor_train.drop(columns="answers"), answers_df],
    axis=1
)

df_reclor_train.head(10)

,context,question,label,id_string,answer_a,answer_b,answer_c,answer_d
0,"In rheumatoid arthritis, the body' s immune sy...","The statements above, if true, most strongly s...",1,train_0,Unlike aspirin and other medications that redu...,A patient treated with the new medication for ...,Joint diseases other than rheumatoid arthritis...,The benefits to rheumatoid arthritis sufferers...
1,Patient: Pharmacists maintain that doctors sho...,The patient's argument proceeds by,0,train_1,attempting to discredit a position by question...,rejecting a questionable position on the groun...,pointing out an unstated assumption on which t...,asserting that pharmacists lack the appropriat...
2,Paula will visit the dentist tomorrow morning ...,The pattern of reasoning displayed above most ...,1,train_2,"If Marge goes to the bank today, Lauren will n...",Kevin will wash his car tomorrow only if Britt...,Renee will do her homework tonight if there is...,Maddie will plan a picnic only if one of her f...
3,Some theorists argue that literary critics sho...,The argument's conclusion follows logically if...,1,train_3,Any critic who is able to help readers make th...,If it is impossible to produce completely valu...,The less readers understand the meaning of a l...,Critics are more likely to provide criticisms ...
4,Shipping Clerk: The five specially ordered shi...,"If the shipping clerk's statements are true, w...",0,train_4,At least one of the shipments sent to Truax Co...,At least one of last week's specially ordered ...,At least one of the shipments sent to Truax Co...,At least one of the shipments sent to Truax Co...
5,"Firms adopting ""profit-related-pay"" (PRP) cont...","If, on the basis of the evidence above, it is ...",0,train_5,Many firms in the metalworking industry have m...,Under PRP contracts costs other than labor cos...,Because introducing PRP contracts greatly chan...,In firms in the metalworking industry where PR...
6,Astronomer: Mount Shalko is the perfect site f...,"Which one of the following, if true, most weak...",3,train_6,More than a dozen insect and plant species end...,The building of the observatory would not caus...,A survey conducted by a team of park rangers c...,Having a complex that covers most of the summi...
7,Opponents of peat harvesting in this country a...,"Which one of the following, if true, most stre...",1,train_7,The peat-harvesting industry of Ireland has be...,The original ecology of the peat-harvesting ar...,"Over hundreds of years, the ecological balance...",The activities of other industries in coming y...
8,Shortly after the power plant opened and began...,"Each of the following, if true, could explain ...",1,train_8,The discharged water was warmer than the norma...,"Because of the new plant, access to the part o...","Because of the new plant, the level of noise d...",The discharged wastewater made the river more ...
9,Any sale item that is purchased can be returne...,"If the statements above are true, which one of...",2,train_9,Any item that is not on sale cannot be returne...,Some construction tools are not returnable for...,No piece of gardening equipment is returnable ...,None of the things that are returnable for a r...


In [68]:
len(df_reclor_train)

4638

In [66]:
df_reclor_train.head(10).to_csv('reclor_sample.csv')

# TweetQA

优点：

- 结构化 CSV 数据

缺点：

- 非选择 QA 模型，需要 Seq 2 Seq 架构，技术难度高
- 包含非 UTF-8 字符

In [33]:
import pandas as pd

BASE_PATH = Path('./TweetQA')
tweet_train_path = BASE_PATH / 'train.csv'
tweet_test_path = BASE_PATH / 'test.csv'

tweet_train = pd.read_csv(tweet_train_path)
tweet_test = pd.read_csv(tweet_test_path)

In [ ]:
tweet_train.head(5)


,Question,Answer,Tweet,qid
0,at which school were first responders on the s...,['independence high school'],"Our prayers are with the students, educators &...",0c871b7e5320d0816d5b2979d67c2649
1,what website is linked in the tweet?,['instagram'],I can finally say it out loud and proud: I'm g...,02505974856f47bbe8b487c57778de1c
2,when does he say kaine is able?,"['5:09 pm- jul 22, 2016.']",KAINE IS ABLE!!!— Cory Booker (@CoryBooker) Ju...,d16eb85d141d5a87bfbc438afbcf50aa
3,what is up to us to keep doing?,['exploring'],Cassini showed us the beauty of Saturn. It rev...,e6e40c7aa9a94b6bab363812ea1918b0
4,where can you see girls doing that?,['at the bar'],"Nicki your video was just you twerking, I see ...",44041dedd11431be77e4a4cf5a8d0276


In [63]:
tweet_train['Tweet'][0]

'Our prayers are with the students, educators & families at Independence High School & all the first responders on the scene. #PatriotPride— Doug Ducey (@dougducey) February 12, 2016'

In [65]:
tweet_train.head(5).to_csv('tweet_sample.csv')